# 08 — SDKB 데이터셋 구조·통계·샘플 문서 둘러보기

현재 `/data` 아래에 구축된 산출물을 한눈에 보기 위한 노트북입니다.

**다루는 내용**
1. 파일 레이아웃 (디렉터리·산출물 한 줄 설명)
2. SDKB 온톨로지 (`nodes.parquet`, `edges.parquet`) 통계
3. 반도체 거절 출원 코호트 (`rejected_patents_meta.parquet`) 통계
4. 인용 선행기술 (`prior_art_edges.parquet`, `prior_art_pairs.parquet`) 통계
5. 샘플 — **거절된 출원문서** (target patent)
6. 샘플 — **거절결정서** (structured + 원본 텍스트)
7. 샘플 — **인용발명(선행기술)** 전문
8. 샘플 — Ground-Truth 증거 매핑 (`§29② 진보성` 인용 evidence_v2)

전제: `make ingest-sirp && make sirp-pairs` 가 한 번 이상 실행되어 parquet 산출물이 존재해야 합니다.

In [ ]:
from __future__ import annotations
import json
from pathlib import Path
import pandas as pd

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 160)

ROOT = Path.cwd().resolve()
if (ROOT / 'data').is_dir():
    pass
elif (ROOT.parent / 'data').is_dir():
    ROOT = ROOT.parent
else:
    raise SystemExit('repo root not found — run from sdkb/ or sdkb/notebooks/')

DATA = ROOT / 'data'
PATENTS = DATA / 'patents'
print('repo root :', ROOT)
print('data dir  :', DATA)

## 1. 파일 레이아웃

| 경로 | 한 줄 설명 |
|---|---|
| `data/semiconductor_v0_3.json` | SDKB v0.3 온톨로지 JSON (nodes·edges·synonyms·cases·provenance) |
| `data/nodes.parquet` / `edges.parquet` | 온톨로지의 노드·엣지 평탄화본 |
| `data/schema_report.json` | 온톨로지 통계·무결성 리포트 |
| `data/patents/raw/semiconductor_industry_rejected_patents.jsonl` | 원본 코호트 (1,000건) |
| `data/patents/rejected_patents_meta.parquet` | 거절 출원 메타 (1,000 × 35) |
| `data/patents/fulltext_corpus.parquet` | 선행기술 본문 코퍼스 (3,154 × 9) |
| `data/patents/ipc_links.parquet` | 출원-IPC 링크 |
| `data/patents/prior_art_edges.parquet` | 거절 인용 엣지 (target → cited) |
| `data/patents/prior_art_pairs.parquet` | 학습/평가용 페어 (라벨·난이도) |
| `data/patents/fulltext/prior_arts/*.txt` | 인용 선행기술 전문 (3,155건) |
| `data/patents/fulltext/etching_prior_arts/*.txt` | Etching 도메인 보강 (192건) |
| `data/patents/rejection_decisions/structured/*.json` | 거절결정서 구조화 결과 (441건) |
| `data/patents/rejection_decisions/_index.jsonl` | 거절결정서 인덱스 |
| `data/patents/ingest_report.json` | ingest 단계 통계 리포트 |

In [ ]:
# 실제 파일이 존재하는지 점검
expected = [
    DATA / 'semiconductor_v0_3.json',
    DATA / 'nodes.parquet',
    DATA / 'edges.parquet',
    DATA / 'schema_report.json',
    PATENTS / 'raw' / 'semiconductor_industry_rejected_patents.jsonl',
    PATENTS / 'rejected_patents_meta.parquet',
    PATENTS / 'fulltext_corpus.parquet',
    PATENTS / 'ipc_links.parquet',
    PATENTS / 'prior_art_edges.parquet',
    PATENTS / 'prior_art_pairs.parquet',
    PATENTS / 'fulltext' / 'prior_arts',
    PATENTS / 'fulltext' / 'etching_prior_arts',
    PATENTS / 'rejection_decisions' / 'structured',
    PATENTS / 'rejection_decisions' / '_index.jsonl',
    PATENTS / 'ingest_report.json',
]
rows = []
for p in expected:
    if p.is_dir():
        n = sum(1 for _ in p.iterdir())
        rows.append({'path': str(p.relative_to(ROOT)), 'exists': True, 'entries': n, 'size_kb': None})
    elif p.exists():
        rows.append({'path': str(p.relative_to(ROOT)), 'exists': True, 'entries': None,
                     'size_kb': round(p.stat().st_size / 1024, 1)})
    else:
        rows.append({'path': str(p.relative_to(ROOT)), 'exists': False, 'entries': None, 'size_kb': None})
pd.DataFrame(rows)

## 2. SDKB 온톨로지 통계

`schema_report.json` 의 무결성 결과와 `nodes/edges.parquet` 분포를 함께 본다.

In [ ]:
schema_report = json.loads((DATA / 'schema_report.json').read_text(encoding='utf-8'))
print('version :', schema_report['version'], '/', schema_report['version_id'])
print('status  :', schema_report['status'])
print('counts  :', schema_report['counts'])
print('integrity.referential.total_dangling =', schema_report['integrity']['referential']['total_dangling'])

nodes = pd.read_parquet(DATA / 'nodes.parquet')
edges = pd.read_parquet(DATA / 'edges.parquet')
print('\nnodes.parquet:', nodes.shape, ' | edges.parquet:', edges.shape)
print('\nNode type distribution:')
print(nodes['type'].value_counts().to_string())
print('\nPredicate distribution:')
print(edges['predicate'].value_counts().to_string())

In [ ]:
# 노드 샘플 (FailureMode 5건)
nodes[nodes['type'] == 'FailureMode'][['id', 'canonical_name', 'description']].head(5)

## 3. 거절 출원 코호트 통계 (1,000건)

In [ ]:
ingest = json.loads((PATENTS / 'ingest_report.json').read_text(encoding='utf-8'))
print('source       :', ingest['source_file'])
print('n_patents    :', ingest['n_patents'])
print('n_ipc_links  :', ingest['n_ipc_links'])
print('n_prior_art_edges:', ingest['n_prior_art_edges'])
print('n_edges_by_source:', ingest['n_edges_by_source'])
print('expanded_schema  :', ingest['expanded_schema'])

In [ ]:
meta = pd.read_parquet(PATENTS / 'rejected_patents_meta.parquet')
print('shape:', meta.shape, '\n')
print('columns:', list(meta.columns), '\n')

print('출원국(office) 분포:')
print(meta['patent_office'].value_counts().to_string(), '\n')

print('process_family 분포 (상위 10):')
print(meta['process_family'].value_counts().head(10).to_string(), '\n')

print('value_chain 분포:')
print(meta['value_chain'].value_counts().to_string(), '\n')

print('primary_ipc_section 분포:')
print(meta['primary_ipc_section'].value_counts().to_string())

In [ ]:
# 출원일(filing_date) 연도 분포
yr = pd.to_datetime(meta['filing_date'], errors='coerce').dt.year
print('출원 연도 분포 (최근 10년):')
print(yr.value_counts().sort_index().tail(10).to_string())

## 4. 인용 선행기술 통계

In [ ]:
pa_edges = pd.read_parquet(PATENTS / 'prior_art_edges.parquet')
pa_pairs = pd.read_parquet(PATENTS / 'prior_art_pairs.parquet')
corpus   = pd.read_parquet(PATENTS / 'fulltext_corpus.parquet')

print('prior_art_edges:', pa_edges.shape)
print('prior_art_pairs:', pa_pairs.shape)
print('fulltext_corpus:', corpus.shape, '\n')

print('인용국가 분포 (edges):')
print(pa_edges['cited_country'].value_counts().to_string(), '\n')

print('source_type (edges):')
print(pa_edges['source_type'].value_counts().to_string(), '\n')

print('legal_basis (edges, 상위 10):')
print(pa_edges['legal_basis'].astype(str).value_counts().head(10).to_string(), '\n')

print('pairs label / difficulty:')
print(pd.crosstab(pa_pairs['label'], pa_pairs['difficulty']))

In [ ]:
# 코퍼스 본문 길이 분포
print('fulltext_corpus.has_content:')
print(corpus['has_content'].value_counts().to_string(), '\n')
print('n_chars 분포 (요약):')
print(corpus['n_chars'].describe().to_string())

## 5. 샘플 — 거절된 출원문서 (target patent)

원본 JSONL 1건을 펼쳐서 출원의 핵심 메타데이터, 청구항, 패밀리 정보를 보여준다.

In [ ]:
raw_jsonl = PATENTS / 'raw' / 'semiconductor_industry_rejected_patents.jsonl'

def load_first_with_rejection(n_scan: int = 50) -> dict:
    """거절결정서 구조화가 있는 첫 레코드를 반환 (없으면 첫 레코드)."""
    fallback = None
    with raw_jsonl.open(encoding='utf-8') as f:
        for i, line in enumerate(f):
            rec = json.loads(line)
            if fallback is None:
                fallback = rec
            rd = rec.get('meta', {}).get('rejection_decision') or {}
            if rd.get('structured_path'):
                return rec
            if i >= n_scan:
                break
    return fallback

sample = load_first_with_rejection()
tp = sample['target_patent']
print('▶ 출원번호 :', tp.get('application_number'))
print('▶ 제목     :', tp.get('title'))
print('▶ 출원일/공개일/등록:', tp.get('date'))
print('▶ IPC      :', tp.get('ipc'))
print('▶ legal_status :', tp.get('legal_status'))
print('\n[ Abstract ]')
print((tp.get('abstract') or '')[:600], '...')
print('\n[ Claim 1 ]')
print((tp.get('claim1') or '')[:600], '...')

In [ ]:
# 전체 청구항(claims_full) 개수와 첫 3개 미리보기
claims_full = tp.get('claims_full') or []
print(f'청구항 총 {len(claims_full)}개\n')
for c in claims_full[:3]:
    if isinstance(c, dict):
        print(f"  - claim {c.get('claim_no')}: {str(c.get('text',''))[:200]}...")
    else:
        print('  -', str(c)[:200], '...')

In [ ]:
# 패밀리(family) 요약 — {publication_numbers: [...], source: ...} 형태
fam = tp.get('family') or {}
if isinstance(fam, dict):
    pubs = fam.get('publication_numbers') or []
    src  = fam.get('source')
else:
    pubs, src = (fam or []), None
print(f'패밀리 멤버 {len(pubs)}건  (source={src})')
for m in pubs[:5]:
    print('  •', m)

## 6. 샘플 — 거절결정서 (rejection decision)

- `structured/*.json`: legal_bases, target_claims, decision_date 등을 파싱한 JSON
- `_index.jsonl`: 출원번호 → pdf/txt/struct 경로 매핑
- (참고) 원본 PDF·OCR txt 는 `data/processed/rejection_decisions/...` 에 위치 — 노트북에서는 구조화 결과를 본다.

In [ ]:
rd_dir = PATENTS / 'rejection_decisions' / 'structured'
rd_index = PATENTS / 'rejection_decisions' / '_index.jsonl'
print('structured json 개수 :', sum(1 for _ in rd_dir.glob('*.json')))

# 인덱스 첫 3건
with rd_index.open(encoding='utf-8') as f:
    idx_rows = [json.loads(l) for l in f]
print('index 총', len(idx_rows), '건')
pd.DataFrame(idx_rows).head(3)

In [ ]:
# sample 출원과 매칭되는 거절결정서 구조화 결과 보기
app_no = str(tp.get('application_number'))
rd_path = rd_dir / f'{app_no}.json'
if not rd_path.exists():
    # 첫 번째로 존재하는 구조화 결과 사용
    rd_path = next(rd_dir.glob('*.json'))
    app_no = rd_path.stem
    print('(sample target에 매칭 없음 → 다른 구조화 결과 사용:', app_no, ')')

rd = json.loads(rd_path.read_text(encoding='utf-8'))
print('▶ application_number :', rd.get('application_number'))
print('▶ decision_date      :', rd.get('decision_date'))
print('▶ ocr_method         :', rd.get('ocr_method'))
print('▶ text_length        :', rd.get('text_length'))
print('▶ legal_bases        :', rd.get('legal_bases'))
print('▶ target_claims      :', rd.get('target_claims'))
print('▶ cited_evidence_phrases (상위 5):')
for p in (rd.get('cited_evidence_phrases') or [])[:5]:
    print('   -', p)
print('▶ cited_evidence_map keys:', list((rd.get('cited_evidence_map') or {}).keys())[:10])

## 7. 샘플 — 인용발명(선행기술) 전문

In [ ]:
pa_dir = PATENTS / 'fulltext' / 'prior_arts'
files = sorted(p for p in pa_dir.glob('*.txt'))
print('prior_arts/*.txt 개수:', len(files))

# 인덱스 메타 한 줄 보기
pa_index = json.loads((pa_dir / '_index.json').read_text(encoding='utf-8'))
print('schema:', pa_index['schema_version'])
print('counts:', pa_index['counts'])
print('by_country:', pa_index['by_country'])

In [ ]:
# Ground-truth 인용 1건의 전문 미리보기
gt_v2 = sample['meta'].get('ground_truth_evidence_v2') or []
cited_id = gt_v2[0]['cited_id'] if gt_v2 else None
if cited_id:
    pa_path = pa_dir / f'{cited_id}.txt'
else:
    pa_path = files[0]

if not pa_path.exists():
    pa_path = files[0]

print('▶ 인용발명 파일:', pa_path.name, '\n')
text = pa_path.read_text(encoding='utf-8')
print(text[:1500], '\n...\n(총', len(text), '자)')

## 8. Ground-Truth 증거 매핑 (`evidence_v2`)

심사관이 거절결정서 본문에서 `§29② 진보성` 등을 들어 인용한 선행기술과 대상 청구항 매핑. 이 구조가 PriorArt 평가의 **gold label** 이다.

In [ ]:
if gt_v2:
    df = pd.DataFrame(gt_v2)
    print('대상 출원:', tp.get('application_number'), '/', tp.get('title'))
    display = df[['cited_id', 'legal_basis', 'target_claims', 'evidence_phrase_no']]
    print(display.to_string(index=False))
else:
    print('샘플에 evidence_v2 가 비어 있음 — 다음 셀에서 전체 통계 확인')

In [ ]:
# 전체 코호트에서 evidence_v2 보유율
n_with_v2 = 0
n_total = 0
with raw_jsonl.open(encoding='utf-8') as f:
    for line in f:
        n_total += 1
        rec = json.loads(line)
        if rec.get('meta', {}).get('ground_truth_evidence_v2'):
            n_with_v2 += 1
print(f'evidence_v2 보유: {n_with_v2} / {n_total} ({100*n_with_v2/n_total:.1f}%)')
print('(ingest_report 기준 270건과 일치해야 함)')

## 다음 단계

- 본문 기반 검색 baseline → [04_prior_art_baseline.ipynb](04_prior_art_baseline.ipynb)
- SDKB ABox 매칭 / SPARQL → [06_sparql_abox_matching.ipynb](06_sparql_abox_matching.ipynb), [07_sparql_prior_art_ontology.ipynb](07_sparql_prior_art_ontology.ipynb)
- 데이터셋 카드: [docs/dataset/cards](../docs/dataset/cards)